# Depedencies

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torch.utils.data import Subset

import numpy as np
# import matplotlib.pyplot as plt
# import joblib

import os
# import zipfile
from pathlib import Path

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Current device: {device}")

# CLEnet

In [ ]:
# EMA1d
class EMA1D(nn.Module):
    """Applies Efficient Multi Scale Attention (EMA) over 1D sequence data.

    Groups channels and applies intra-group attention mechanisms
    that include channel-wise convolution, group normalization, and dual-path
    cross-attention to enhance feature representations in temporal data.

    Args:
        channels (int): Number of input channels.
        factor (int): Number of groups to divide the channels into. Must divide `channels` evenly.
    """
    def __init__(self, channels, factor=32):
        super(EMA1D, self).__init__()
        self.groups = factor
        assert channels // self.groups > 0
        assert channels % self.groups == 0, f"channels ({channels}) must be divisible by factor/groups ({self.groups})"

        self.softmax = nn.Softmax(-1)
        self.agp = nn.AdaptiveAvgPool1d(1)
        self.pool_c = nn.AdaptiveAvgPool1d(1) # Pool across sequence length
        self.gn = nn.GroupNorm(channels // self.groups, channels // self.groups)
        self.conv1x1 = nn.Conv1d(channels // self.groups, channels // self.groups, kernel_size=1, stride=1, padding=0)
        self.conv3 = nn.Conv1d(channels // self.groups, channels // self.groups, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        b, c, l = x.size()  # (b, c, l)
        group_x = x.reshape(b * self.groups, -1, l) # b*g, c//g, l

        # Apply channel-wise avg pool across length
        x_c = self.pool_c(group_x) # b*g, c//g, 1

        # Apply 1x1 conv to channel-pooled features
        c_att = self.conv1x1(x_c) # b*g, c//g, 1

        x1 = self.gn(group_x * c_att.sigmoid())
        x2 = self.conv3(group_x)

        # Cross-attention between x1 & x2
        # Path 1: x1 attends to x2
        x11 = self.softmax(self.agp(x1).reshape(b * self.groups, -1, 1).permute(0, 2, 1)) # b*g, 1, c//g
        x12 = x2.reshape(b * self.groups, c // self.groups, -1) # b*g, c//g, l

        # Path 2: x2 attends to x1
        x21 = self.softmax(self.agp(x2).reshape(b * self.groups, -1, 1).permute(0, 2, 1)) # b*g, 1, c//g
        x22 = x1.reshape(b * self.groups, c // self.groups, -1) # b*g, c//g, l

        # Combine attention weights
        weights = (torch.matmul(x11, x12) + torch.matmul(x21, x22)).reshape(b * self.groups, 1, l)

        return (group_x * weights.sigmoid()).reshape(b, c, l) # (b, c, l)

# Conv Block

class CNN_EMA1D_Block(nn.Module):
    """A convolutional block that combines 1D convolution, ReLU activation,
    EMA-1D, and optional pooling or dropout.

    Args:
        in_channels (int): Number of input channels.
        out_channels (int): Number of output channels after convolution.
        kernel_size (int): Size of the convolutional kernel.
        stride (int): Stride of the convolution.
        padding (int): Padding added to both sides of the input.
        factor (int): Number of groups for the EMA1D attention mechanism.
        kernel_pool (int, optional): Kernel size for AvgPool1d (if pooling is used).
        stride_pool (int, optional): Stride for AvgPool1d (if pooling is used).
        pool_dropout (str, optional): If 'pool', applies average pooling;
                                      if 'dropout', applies dropout; if None, applies neither.
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding, factor, kernel_pool=None, stride_pool=None, pool_dropout=None):
        super(CNN_EMA1D_Block, self).__init__()

        # CNN_EMA1D Block
        self.conv = nn.Conv1d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=stride, padding=padding)
        self.relu = nn.ReLU() # Activation function
        self.ema1d = EMA1D(channels=out_channels, factor=factor) # EMA-1d

        # Conditional to add either average pooling or dropout
        self.pool_dropout = pool_dropout
        if pool_dropout == 'pool':
            assert kernel_pool is not None and stride_pool is not None, "Pooling selected but kernel_pool or stride_pool is None"

            self.pool = nn.AvgPool1d(kernel_size=kernel_pool, stride=stride_pool)
        elif pool_dropout == 'dropout':
            self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):
        x = self.conv(x)
        x = self.relu(x)
        x = self.ema1d(x)

        if self.pool_dropout == 'pool':
            x = self.pool(x)
        elif self.pool_dropout == 'dropout':
            x = self.dropout(x)

        return x # b, c, l

# Conv_Block + EMA1d
class CNN_EMA1D(nn.Module):
    """A stacked 1D convolutional network with EMA-1D attention applied at each layer.

    This module constructs a sequence of CNN_EMA1D_Block layers, where each block applies
    convolution, ReLU, EMA1D attention, and optionally average pooling or dropout.

    Args:
        channel_plan (List[int]): List defining the number of channels in each layer.
                                  Must have at least two elements: input and one output.
        kernel_size (int): Kernel size for all Conv1d layers.
        stride (int): Stride for all Conv1d layers.
        padding (int): Padding for all Conv1d layers.
        factor (int): Number of groups for the EMA1D attention mechanism.
        kernel_pool (int or None): Kernel size for AvgPool1d if pooling is used.
        stride_pool (int or None): Stride for AvgPool1d if pooling is used.
        pool_dropout (str or None): If 'pool', applies AvgPool1d; if 'dropout', applies dropout;
                                    if None, no additional operation is applied.
    """
    def __init__(self, channel_plan, kernel_size, stride, padding, factor, kernel_pool, stride_pool, pool_dropout):
        super(CNN_EMA1D, self).__init__()
        assert len(channel_plan) >= 2, "channel_plan must have at least input and one output"

        # Repeat layers
        self.layers = nn.ModuleList()
        for i in range(len(channel_plan) - 1): # Loop over adjacent pairs in channel_plan
            in_channel = channel_plan[i] # Current input channel
            out_channel = channel_plan[i + 1] # Next output channel
            self.layers.append(
                CNN_EMA1D_Block(
                    in_channels=in_channel,
                    out_channels=out_channel,
                    kernel_size=kernel_size,
                    stride=stride,
                    padding=padding,
                    factor=factor,
                    kernel_pool=kernel_pool,
                    stride_pool=stride_pool,
                    pool_dropout=pool_dropout
                )
            )

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x # (b, c, l)

# Extracting features
class Features_Block(nn.Module):
    """A feature extraction block combining stacked CNN_EMA1D layers followed by
    additional Conv1d, ReLU, EMA1D attention, and dropout layers.

    Args:
        channel_plan (List[int]): Channel sizes for the CNN_EMA1D block layers.
        kernel_size (int): Kernel size for all convolutional layers.
        stride (int): Stride for all convolutional layers.
        padding (int): Padding for all convolutional layers.
        factor (int): Number of groups for the EMA1D attention mechanism.
        kernel_pool (int or None): Kernel size for average pooling in CNN_EMA1D blocks.
        stride_pool (int or None): Stride for average pooling in CNN_EMA1D blocks.
        pool_dropout (str or None): Specifies pooling or dropout in CNN_EMA1D blocks.
        cnn_plan (List[int]): Channel sizes for the final convolutional layers after CNN_EMA1D.
    """
    def __init__(self, channel_plan, kernel_size, stride, padding, factor, kernel_pool, stride_pool, pool_dropout, cnn_plan):
        super(Features_Block, self).__init__()

        # Ensure that output_channel in channel_plan matches input_channel from cnn_plan
        assert channel_plan[-1] == cnn_plan[0], (
            f"Channel mismatch: Expected cnn_plan[0] ({cnn_plan[0]}) to match channel_plan[-1] ({channel_plan[-1]})"
        )
        assert len(cnn_plan) == 3, (f"cnn_plan must be list[int] length of 3")

        # CNN_EMA1D block (morphological feature extraction)
        self.cnn_ema1d = CNN_EMA1D(
            channel_plan,
            kernel_size,
            stride,
            padding,
            factor,
            kernel_pool,
            stride_pool,
            pool_dropout
        )

        # Sequential CNN + ReLU + EMA + Dropout block
        self.conv_1 = nn.Conv1d(in_channels=cnn_plan[0], out_channels=cnn_plan[1], kernel_size=kernel_size, stride=stride, padding=padding)
        self.relu_1 = nn.ReLU()
        self.conv_2 = nn.Conv1d(in_channels=cnn_plan[1], out_channels=cnn_plan[2], kernel_size=kernel_size, stride=stride, padding=padding)
        self.relu_2 = nn.ReLU()
        self.ema1d = EMA1D(channels=cnn_plan[2], factor=factor) # EMA-1d
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.cnn_ema1d(x)
        x = self.relu_1(self.conv_1(x))
        x = self.relu_2(self.conv_2(x))
        x = self.ema1d(x)
        x = self.dropout(x)

        return x # (b, c, l)


# Temporal feature

class Temporal(nn.Module):
    """Temporal feature extractor module using fully connected layers followed by an LSTM.

    Args:
        in_channel (int): Number of input channels (should match the output channels of the preceding CNN).
        num_layers (int): Number of layers in the LSTM.
    """
    def __init__(self, in_channel, num_layers): # in_channel should be the same as cnn_plan[-1]
        super(Temporal, self).__init__()

        self.fc_1 = nn.Linear(in_features=in_channel, out_features=1024)
        self.fc_2 = nn.Linear(in_features=1024, out_features=512)

        self.lstm = nn.LSTM(
            input_size=512,
            hidden_size=512,
            num_layers=num_layers,
            batch_first=True
        )

    def forward(self, x):
        x = x.permute(0, 2, 1) # (b, c, l) -> (b, l, c)
        x = self.fc_1(x)
        x = self.fc_2(x)

        x, _ = self.lstm(x)

        return x


class CLEnet(nn.Module):
    """CNN, LSTM, EMA-1D (CLEnet)

    This model is designed to process EEG data by extracting spatial features
    using convolutional blocks (`Features_Block`), capturing temporal dependencies
    using LSTM blocks (`Temporal`), and finally producing a denoised EEG output
    of the same shape as the original input using a fully connected layer.

    Args:
        channel_plan (list[int]): Configuration of channels for convolutional layers.
        kernel_sizes (list[int]): Two kernel sizes for dual-path convolutional branches.
        stride (int): Stride used in the convolutional layers.
        padding (int): Padding applied to the convolutional layers.
        factor (int): Expansion or bottleneck factor used in `Features_Block`.
        kernel_pool (int): Kernel size used in pooling layers.
        stride_pool (int): Stride used in pooling layers.
        pool_dropout (float): Dropout rate applied after pooling.
        cnn_plan (list[int]): Configuration for CNN layer channel dimensions.
        num_layers (int): Number of LSTM layers in each `Temporal` block.
        out_features (int): The number of output features (should match the length of the original EEG signal).

    """
    def __init__(self, channel_plan, kernel_sizes, stride, padding, factor, kernel_pool, stride_pool, pool_dropout, cnn_plan, num_layers, in_features, out_features):
        super(CLEnet, self).__init__()
        assert len(kernel_sizes) == 2, "kernel_sizes must be a list[int] of two elements"

        # Create two feature extractors with different kernel sizes
        self.features = nn.ModuleList([
            Features_Block(
                channel_plan=channel_plan,
                kernel_size=k,
                stride=stride,
                padding=padding,
                factor=factor,
                kernel_pool=kernel_pool,
                stride_pool=stride_pool,
                pool_dropout=pool_dropout,
                cnn_plan=cnn_plan
            ) for k in kernel_sizes
        ])

        # Create two temporal extractors
        self.temporals = nn.ModuleList([
            Temporal(in_channel=cnn_plan[-1], num_layers=num_layers)
            for _ in range(2)
        ])

        # out_features = length of original EEG signal
        self.fc = nn.Linear(in_features=in_features, out_features=out_features)

    def forward(self, x):
        # Apply each feature and temporal extractor in parallel
        x_1 = self.temporals[0](self.features[0](x)) # (batch, length, hidden_size)
        x_2 = self.temporals[1](self.features[1](x)) # (b, l, h)

        # Transpose l & h to be able to concat
        x_1 = x_1.transpose(1, 2) # (b, h, l)
        x_2 = x_2.transpose(1, 2) # (b, h, l)

        x = torch.cat([x_1, x_2], dim=-1) # (b, l, h*2)
        x = x.reshape(x.size(0), -1) # (b, l * h*2)

        x = self.fc(x)
        x = x.unsqueeze(1) # (b, 1, sequence_length)

        return x

# Dataset & Dataloader

In [ ]:
class EEG_Dataset(Dataset):
    """
    A PyTorch Dataset for loading raw and clean EEG epoch pairs.
    It expects the following structure:
    - data/
        - raw/
            - raw_training_epochs/
                - subject1/
                    - c3_epoch0_raw.pt
        - clean/
            - clean_training_epochs/
                - subject1/
                    - c3_epoch0_clean.pt
    """
    def __init__(self, data_dir, split):
        """
        Args:
            data_dir (str): The path to the root 'data' directory.
            split (str): The dataset split (e.g., 'training_epochs', 'validation_epochs', or 'test_epochs').
        """
        # Construct the paths to the raw and clean data folders for the specified split
        self.raw_dir = Path(data_dir) / 'raw' / f'raw_{split}'
        self.clean_dir = Path(data_dir) / 'clean' / f'clean_{split}'

        if not self.raw_dir.is_dir() or not self.clean_dir.is_dir():
            raise FileNotFoundError(f"One of the specified directories does not exist: {self.raw_dir} or {self.clean_dir}")

        self.file_pairs = []

        # Traverse the directory to find all raw files and their corresponding clean files
        for subject_dir in self.raw_dir.iterdir():
            if subject_dir.is_dir():
                for raw_file_path in subject_dir.glob('*.pt'):
                    # The clean file path is found by replacing the directory and file suffix
                    relative_path = raw_file_path.relative_to(self.raw_dir)
                    clean_file_path = self.clean_dir / relative_path.with_name(
                        raw_file_path.stem.replace('_raw', '_clean') + '.pt'
                    )

                    if clean_file_path.is_file():
                        self.file_pairs.append((raw_file_path, clean_file_path))
                    # else:
                    #     print(f"Warning: Corresponding clean file not found for {raw_file_path}")

    def __len__(self):
        """Returns the total number of data samples."""
        return len(self.file_pairs)

    def __getitem__(self, idx):
      """Loads and returns a raw and clean pair, normalized to [-1, 1]."""
      raw_path, clean_path = self.file_pairs[idx]

      # Load the tensors from their file paths
      raw_tensor = torch.load(raw_path)
      clean_tensor = torch.load(clean_path)

      # Add channel dim: [512] -> [1, 512]
      raw_tensor = raw_tensor.unsqueeze(0)
      clean_tensor = clean_tensor.unsqueeze(0)

      # Standardize each sample to [-1, 1]
      def normalize(tensor):
          min_val = tensor.min()
          max_val = tensor.max()
          if max_val > min_val:  # Avoid division by zero
              tensor = 2 * (tensor - min_val) / (max_val - min_val) - 1
          else:
              tensor = torch.zeros_like(tensor)
          return tensor

      raw_tensor = normalize(raw_tensor)
      clean_tensor = normalize(clean_tensor)

      return raw_tensor, clean_tensor



# Dataset & Loaders

# ===== Training =====
train_dataset = EEG_Dataset(
    "/kaggle/input/eeg-clean-raw/dat-dataset-2-pro-max-supreme", 
    "training_epochs"
)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)


# ===== Valid =====
valid_dataset = EEG_Dataset(
    "/kaggle/input/eeg-clean-raw/dat-dataset-2-pro-max-supreme",
    "validation_epochs"
)
valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=64, shuffle=False)


# ===== Test =====
test_dataset = EEG_Dataset(
    "/kaggle/input/eeg-clean-raw/dat-dataset-2-pro-max-supreme",
    "testing_epochs"
)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

# Training

In [ ]:
# ===== Hyperparameters =====
channel_plan = [1, 16, 32, 128, 256]
kernel_sizes = [3, 5]
stride = 1
padding = 1
factor = 8
kernel_pool = 2
stride_pool = 2
pool_dropout = 'pool'
cnn_plan = [256, 512, 512]
num_layers = 1
in_features = 29696
out_features = 512
learning_rate = 1e-4


# ===== Model, Loss, Optimizer =====
model = CLEnet(
    channel_plan=channel_plan,
    kernel_sizes=kernel_sizes,
    stride=stride,
    padding=padding,
    factor=factor,
    kernel_pool=kernel_pool,
    stride_pool=stride_pool,
    pool_dropout=pool_dropout,
    cnn_plan=cnn_plan,
    num_layers=num_layers,
    in_features=in_features,
    out_features=out_features
)

model = model.to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, betas=(0.05, 0.9))

num_epochs = 200
train_loss_history = []
val_loss_history = []
start_epoch = 0


# ===== Early Stoppage Parameters =====
patience = 20
best_val_loss = float('inf')
epochs_no_improve = 0
best_model_state = None


# ===== Paths for saving =====
checkpoint_dir = '/kaggle/working/CLEnet'
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(checkpoint_dir, 'clenet_checkpoint.pth')

# ===== Resume if model checkpoint exists =====
start_epoch = 0
train_loss_history = []
val_loss_history = []

if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    train_loss_history = checkpoint['train_loss_history']
    val_loss_history = checkpoint['val_loss_history']
    print(f"Resumed from epoch {start_epoch}")


# ===== Training =====
for epoch in range(start_epoch, num_epochs):
    model.train()
    train_running_loss = 0.0

    for batch_idx, (raw, clean) in enumerate(train_loader):
        raw = raw.to(device)
        clean = clean.to(device)

        optimizer.zero_grad()
        outputs = model(raw)
        loss = criterion(outputs, clean)
        loss.backward()
        optimizer.step()

        train_running_loss += loss.item()

    train_avg_loss = train_running_loss / len(train_loader)
    train_loss_history.append(train_avg_loss)

    # ====== Validation ======
    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for val_raw, val_clean in valid_loader:
            val_raw = val_raw.to(device)
            val_clean = val_clean.to(device)

            val_outputs = model(val_raw)
            val_loss = criterion(val_outputs, val_clean)
            val_running_loss += val_loss.item()

    val_avg_loss = val_running_loss / len(valid_loader)
    val_loss_history.append(val_avg_loss)


    # ===== Early Patience =====
    if val_avg_loss < best_val_loss:
        best_val_loss = val_avg_loss
        epochs_no_improve = 0
        best_model_state = model.state_dict()
    else:
        epochs_no_improve += 1

    print(f"Epoch {epoch +1}/{num_epochs} | No improve: {epochs_no_improve}")
    if epochs_no_improve >= patience:
        print(f"Early stopping after {epoch+1} epochs.")
        break


    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss_history': train_loss_history,
            'val_loss_history': val_loss_history,
            'hyperparameters': {
                'channel_plan': channel_plan,
                'kernel_sizes': kernel_sizes,
                'stride': stride,
                'padding': padding,
                'factor': factor,
                'kernel_pool': kernel_pool,
                'stride_pool': stride_pool,
                'pool_dropout': pool_dropout,
                'cnn_plan': cnn_plan,
                'num_layers': num_layers,
                'in_features': in_features,
                'out_features': out_features,
                'learning_rate': learning_rate,
                'num_epochs': num_epochs
            }
        }, checkpoint_path)
        print(f"Checkpoint saved to: {checkpoint_path}")



# ===== Saving model =====
# model_dir = '/content/drive/MyDrive/trained_models/CLEnet'
model_dir = '/kaggle/working/models'
os.makedirs(model_dir, exist_ok=True)
model_cpu = model.to('cpu') 

# Save the model state dictionary
full_model_path = os.path.join(model_dir, 'clenet.pth')

# Save training history and hyperparameters
torch.save({
    'train_loss_history': train_loss_history,
    'val_loss_history': val_loss_history,
    'hyperparameters': {
        'channel_plan': channel_plan,
        'kernel_sizes': kernel_sizes,
        'epochs': num_epochs,
        'stride': stride,
        'padding': padding,
        'factor': factor,
        'kernel_pool': kernel_pool,
        'stride_pool': stride_pool,
        'pool_dropout': pool_dropout,
        'cnn_plan': cnn_plan,
        'num_layers': num_layers,
        'in_features': in_features,
        'out_features': out_features,
        'learning_rate': learning_rate,
        'num_epochs': num_epochs
    }
}, full_model_path)
print(f"Training history saved to: {full_model_path}")

# Metrics

In [ ]:
# Metrics to measure success

def compute_snr(raw, clean):
    noise = raw - clean
    signal_power = np.mean(clean ** 2)
    noise_power = np.mean(noise ** 2)
    snr = 10 * np.log10(signal_power / noise_power)

    return snr

def compute_rrmse_time(raw, clean):
    numerator = np.sqrt(np.mean((raw - clean) ** 2))
    denominator = np.sqrt(np.mean(clean ** 2))
    return numerator / denominator

def compute_rrmse_freq(raw, clean):
    raw_fft = np.abs(np.fft.rfft(raw))
    clean_fft = np.abs(np.fft.rfft(clean))
    numerator = np.sqrt(np.mean((raw_fft - clean_fft) ** 2))
    denominator = np.sqrt(np.mean(clean_fft ** 2))
    return numerator / denominator

def compute_average_cc(raw, clean):
    correlations = []
    for i in range(raw.shape[0]):
        r = np.corrcoef(raw[i], clean[i])[0, 1]
        correlations.append(r)
    return np.mean(correlations)


def evaluate_model_metrics(model, dataloader):
    model.eval()
    snr_list = []
    rrmse_time_list = []
    rrmse_freq_list = []
    cc_list = []

    # Testing
    with torch.no_grad():
        for raw, clean in dataloader:
            # Run model forward
            model_output_batch = model(raw)

            # Move tensors to CPU and convert to numpy
            raw_np = raw.cpu().numpy()
            clean_np = clean.cpu().numpy()
            output_np = model_output_batch.cpu().numpy()

            # Compute metrics per sample in batch
            for i in range(raw_np.shape[0]):
                # raw_sample = raw_np[i]
                clean_sample = clean_np[i]
                model_output_sample = output_np[i]

                # Use model output and clean target (or raw if comparing raw->clean)
                snr_val = compute_snr(model_output_sample, clean_sample)
                rrmse_time_val = compute_rrmse_time(model_output_sample, clean_sample)
                rrmse_freq_val = compute_rrmse_freq(model_output_sample, clean_sample)
                cc_val = compute_average_cc(model_output_sample, clean_sample)

                snr_list.append(snr_val)
                rrmse_time_list.append(rrmse_time_val)
                rrmse_freq_list.append(rrmse_freq_val)
                cc_list.append(cc_val)

    # Aggregate metric results across entire dataset
    metrics = {
        "SNR": np.mean(snr_list),
        "RRMSE_time": np.mean(rrmse_time_list),
        "RRMSE_freq": np.mean(rrmse_freq_list),
        "Average_CC": np.mean(cc_list)
    }
    return metrics

# Testing

In [ ]:
# Printing metrics
metrics = evaluate_model_metrics(model, test_loader)

print("Test metrics:")
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")
    
with open('/kaggle/working/metrics_output.txt', 'w') as file:
    for metric, value in metrics.items():
        file.write(f"{metrics}: {value}")